# Comparative Machine Learning Analysis for Oil Recovery Factor Prediction
## SVR | Random Forest | XGBoost | Gradient Boosting
### 10-Fold Cross-Validation | Comprehensive Statistical & Visual Analysis

In [ ]:
# ── Core dependencies ──────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, shapiro, probplot

# ── ML frameworks ──────────────────────────────────────────────────────────
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import KFold, cross_val_score, learning_curve
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error
)
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
import xgboost as xgb

# ── SHAP ───────────────────────────────────────────────────────────────────
import shap

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Publication style ──────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.labelsize'   : 12,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.linewidth'   : 1.2,
    'xtick.labelsize'  : 10,
    'ytick.labelsize'  : 10,
    'legend.fontsize'  : 10,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

# ── Colour palette ─────────────────────────────────────────────────────────
PALETTE = {
    'SVR'              : '#E64B35',
    'Random Forest'    : '#4DBBD5',
    'XGBoost'          : '#00A087',
    'Gradient Boosting': '#F39B7F',
}
MODEL_NAMES = list(PALETTE.keys())

print('All libraries loaded successfully ✓')

## 1. Data Loading & Exploratory Analysis

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('Proxy6.csv')
df.columns = df.columns.str.strip()

TARGET = 'Oil_recovery_factor (%)'
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f'Dataset shape : {df.shape}')
print(f'Features      : {len(FEATURES)}')
print(f'Target range  : [{y.min():.4f}, {y.max():.4f}] %')
print(f'Missing values: {df.isnull().sum().sum()}')
df.describe().T.style.background_gradient(cmap='Blues').format('{:.4f}')

In [ ]:
# ── Pretty feature labels ──────────────────────────────────────────────────
FEAT_LABELS = {
    'APV'                        : 'APV',
    'Adsorption (ug/g)'          : 'Adsorption\n(µg/g)',
    'Rock_compressibility (1/psi)': 'Rock Comp.\n(1/psi)',
    'FWM'                        : 'FWM',
    'Injection_temperature (°F)' : 'Inj. Temp\n(°F)',
    'Oil_viscosity (cp)'         : 'Oil Visc.\n(cp)',
    'Permeability (md)'          : 'Permeability\n(md)',
    'Polymer_concentration (ppm)': 'Polymer Conc.\n(ppm)',
    'Porosity'                   : 'Porosity',
    'Reservoir_pressure (psi)'   : 'Res. Pressure\n(psi)',
    'RRF'                        : 'RRF',
    'Reservoir_temperature (°F)' : 'Res. Temp\n(°F)',
    'Solution_viscosity (cp)'    : 'Sol. Visc.\n(cp)',
    'Water_salinity (ppm)'       : 'Water Salinity\n(ppm)',
}

# Fix encoding in column names
df.columns = [c.replace('\ufffd', '°') for c in df.columns]
X.columns  = [c.replace('\ufffd', '°') for c in X.columns]
FEATURES   = list(X.columns)

print('Column names cleaned ✓')

## 2. Pearson Correlation Heatmap

In [ ]:
corr = df.corr(method='pearson')

# ── p-value mask ───────────────────────────────────────────────────────────
n = len(df)
pval_mat = pd.DataFrame(np.ones_like(corr), columns=corr.columns, index=corr.index)
for c1 in corr.columns:
    for c2 in corr.columns:
        if c1 != c2:
            _, p = pearsonr(df[c1], df[c2])
            pval_mat.loc[c1, c2] = p

sig_mask = pval_mat > 0.05          # mask non-significant pairs
upper    = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(14, 11))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(
    corr, mask=upper, cmap=cmap, vmax=1, vmin=-1, center=0,
    square=True, linewidths=0.4, cbar_kws={'shrink': .75, 'label': 'Pearson r'},
    annot=True, fmt='.2f', annot_kws={'size': 7.5}, ax=ax
)

# hatching for non-significant cells (lower triangle only)
for i in range(len(corr)):
    for j in range(i):
        if pval_mat.iloc[i, j] > 0.05:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                        hatch='///', color='white', lw=0))

short = [c.split('(')[0].strip().replace('_', ' ') for c in corr.columns]
ax.set_xticklabels(short, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short, rotation=0, fontsize=9)
ax.set_title('Pearson Correlation Matrix\n(hatched = p > 0.05)', pad=15)

plt.tight_layout()
plt.savefig('fig01_pearson_heatmap.png')
plt.show()
print('Figure 1 saved ✓')

## 3. Target Distribution & Outlier Detection

In [ ]:
# ── IQR + Z-score outlier detection ────────────────────────────────────────
Q1, Q3 = y.quantile(0.25), y.quantile(0.75)
IQR = Q3 - Q1
iqr_out  = ((y < Q1 - 1.5*IQR) | (y > Q3 + 1.5*IQR))
z_scores = np.abs(stats.zscore(y))
z_out    = z_scores > 3

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.35)

# --- (a) Histogram + KDE of target
ax0 = fig.add_subplot(gs[0, 0])
sns.histplot(y, bins=40, kde=True, color='#4DBBD5', edgecolor='white',
             line_kws={'lw': 2.5}, ax=ax0)
ax0.axvline(y.mean(), color='#E64B35', lw=2, ls='--', label=f'Mean={y.mean():.2f}')
ax0.axvline(y.median(), color='#00A087', lw=2, ls=':', label=f'Median={y.median():.2f}')
ax0.set_xlabel('Oil Recovery Factor (%)')
ax0.set_ylabel('Count')
ax0.set_title('(a) Target Distribution')
ax0.legend()

# --- (b) Box plot with jitter
ax1 = fig.add_subplot(gs[0, 1])
bp = ax1.boxplot(y, vert=True, patch_artist=True, notch=True,
                 boxprops=dict(facecolor='#4DBBD5', alpha=0.6),
                 medianprops=dict(color='#E64B35', lw=2.5),
                 whiskerprops=dict(lw=1.5), capprops=dict(lw=1.5),
                 flierprops=dict(marker='o', markersize=6, alpha=0.5, markerfacecolor='#E64B35'))
jitter = np.random.normal(1, 0.04, size=len(y))
ax1.scatter(jitter, y, alpha=0.25, s=15, color='#3C5488', zorder=3)
ax1.scatter(jitter[iqr_out], y[iqr_out], color='#E64B35', s=40,
            zorder=4, label=f'IQR outliers (n={iqr_out.sum()})')
ax1.set_xticklabels(['Oil RF (%)'])
ax1.set_ylabel('Oil Recovery Factor (%)')
ax1.set_title('(b) Box Plot + Outliers')
ax1.legend()

# --- (c) Q-Q plot
ax2 = fig.add_subplot(gs[0, 2])
(osm, osr), (slope, intercept, r) = probplot(y, dist='norm')
ax2.scatter(osm, osr, color='#4DBBD5', alpha=0.6, s=20, label='Data')
line_x = np.array([osm.min(), osm.max()])
ax2.plot(line_x, slope*line_x + intercept, color='#E64B35', lw=2, label='Normal line')
stat_w, p_w = shapiro(y[:5000] if len(y) > 5000 else y)
ax2.set_xlabel('Theoretical Quantiles')
ax2.set_ylabel('Sample Quantiles')
ax2.set_title(f'(c) Q-Q Plot  (Shapiro p={p_w:.3f})')
ax2.legend()

# --- (d) Feature-wise outlier counts (IQR)
ax3 = fig.add_subplot(gs[1, :])
out_counts = {}
for col in FEATURES:
    q1, q3 = X[col].quantile(0.25), X[col].quantile(0.75)
    iqr_c = q3 - q1
    out_counts[col] = ((X[col] < q1 - 1.5*iqr_c) | (X[col] > q3 + 1.5*iqr_c)).sum()

out_df = pd.Series(out_counts).sort_values(ascending=False)
colors_bar = ['#E64B35' if v > 0 else '#CCCCCC' for v in out_df.values]
bars = ax3.bar(range(len(out_df)), out_df.values, color=colors_bar, edgecolor='white', lw=0.5)
ax3.set_xticks(range(len(out_df)))
short_feat = [c.split('(')[0].strip().replace('_', ' ') for c in out_df.index]
ax3.set_xticklabels(short_feat, rotation=35, ha='right', fontsize=9)
ax3.set_ylabel('Outlier Count (IQR method)')
ax3.set_title('(d) Per-Feature Outlier Count (IQR 1.5×)')
for bar, val in zip(bars, out_df.values):
    if val > 0:
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(val), ha='center', va='bottom', fontsize=8, fontweight='bold')

fig.suptitle('Target Variable Characterization & Outlier Detection',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('fig02_outlier_detection.png')
plt.show()
print(f'Figure 2 saved ✓  |  IQR outliers in target: {iqr_out.sum()}')

## 4. KDE Plots — All Features

In [ ]:
ncols = 4
nrows = int(np.ceil((len(FEATURES) + 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.2))
axes = axes.flatten()

all_cols = FEATURES + [TARGET]
cmap_kde = plt.cm.get_cmap('tab20', len(all_cols))

for i, col in enumerate(all_cols):
    ax = axes[i]
    data = df[col]
    color = cmap_kde(i)
    sns.kdeplot(data, ax=ax, fill=True, color=color, alpha=0.35, linewidth=2)
    ax.axvline(data.mean(),   color='#E64B35', ls='--', lw=1.5, label='Mean')
    ax.axvline(data.median(), color='#3C5488', ls=':',  lw=1.5, label='Median')
    skew_val = data.skew()
    kurt_val = data.kurtosis()
    label = col.split('(')[0].replace('_', ' ').strip()
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel('')
    ax.text(0.97, 0.95, f'Skew={skew_val:.2f}\nKurt={kurt_val:.2f}',
            transform=ax.transAxes, fontsize=7.5,
            va='top', ha='right', bbox=dict(boxstyle='round,pad=0.3',
            facecolor='white', alpha=0.7, edgecolor='grey'))
    if i == 0:
        ax.legend(fontsize=7)

for j in range(len(all_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Kernel Density Estimation — All Variables',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig03_kde_all_features.png')
plt.show()
print('Figure 3 saved ✓')

## 5. Model Definition & 10-Fold Cross-Validation

In [ ]:
# ── Scale features ─────────────────────────────────────────────────────────
scaler  = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=FEATURES)

# ── Model definitions ──────────────────────────────────────────────────────
models = {
    'SVR': SVR(
        kernel='rbf', C=100, epsilon=0.01, gamma='scale'
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features='sqrt', random_state=SEED, n_jobs=-1
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        reg_lambda=1.0, random_state=SEED, verbosity=0
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, min_samples_split=4, random_state=SEED
    ),
}

# ── 10-fold CV ─────────────────────────────────────────────────────────────
kf = KFold(n_splits=10, shuffle=True, random_state=SEED)

cv_results = {}
oof_preds  = {name: np.zeros(len(y)) for name in models}
fold_metrics = {name: [] for name in models}
trained_models = {}

for name, model in models.items():
    print(f'  Training {name} ...', end=' ')
    Xdata = X_scaled.values if name == 'SVR' else X.values
    r2_list, rmse_list, mae_list, mape_list = [], [], [], []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(Xdata)):
        X_tr, X_val = Xdata[tr_idx], Xdata[val_idx]
        y_tr, y_val = y.values[tr_idx], y.values[val_idx]

        m = model.__class__(**model.get_params())
        m.fit(X_tr, y_tr)
        preds = m.predict(X_val)
        oof_preds[name][val_idx] = preds

        r2_list.append(r2_score(y_val, preds))
        rmse_list.append(np.sqrt(mean_squared_error(y_val, preds)))
        mae_list.append(mean_absolute_error(y_val, preds))
        mape_list.append(mean_absolute_percentage_error(y_val, preds) * 100)

    cv_results[name] = {
        'R2'  : np.array(r2_list),
        'RMSE': np.array(rmse_list),
        'MAE' : np.array(mae_list),
        'MAPE': np.array(mape_list),
    }

    # Fit on full data for SHAP / feature importance
    model.fit(Xdata, y.values)
    trained_models[name] = (model, Xdata)
    print(f'R²={np.mean(r2_list):.4f} ± {np.std(r2_list):.4f}')

print('\nAll models trained ✓')

## 6. Performance Summary Table

In [ ]:
# ── Build summary dataframe ─────────────────────────────────────────────────
rows = []
for name in MODEL_NAMES:
    res = cv_results[name]
    # AIC / BIC (OLS approximation using residuals)
    resid = y.values - oof_preds[name]
    n_obs = len(y)
    k     = len(FEATURES) + 1
    sse   = np.sum(resid**2)
    sigma2 = sse / n_obs
    log_lik = -n_obs/2 * (1 + np.log(2*np.pi*sigma2))
    aic  = 2*k - 2*log_lik
    bic  = k*np.log(n_obs) - 2*log_lik

    rows.append({
        'Model'        : name,
        'R² (mean)'    : f"{res['R2'].mean():.4f}",
        'R² (std)'     : f"{res['R2'].std():.4f}",
        'RMSE (mean)'  : f"{res['RMSE'].mean():.4f}",
        'RMSE (std)'   : f"{res['RMSE'].std():.4f}",
        'MAE (mean)'   : f"{res['MAE'].mean():.4f}",
        'MAE (std)'    : f"{res['MAE'].std():.4f}",
        'MAPE % (mean)': f"{res['MAPE'].mean():.2f}",
        'AIC'          : f"{aic:.1f}",
        'BIC'          : f"{bic:.1f}",
    })

summary_df = pd.DataFrame(rows).set_index('Model')

def highlight_best(s):
    """Green for best, salmon for worst in each column."""
    numeric = s.astype(float)
    higher_is_better = s.name in ['R² (mean)']
    best  = numeric.idxmax() if higher_is_better else numeric.idxmin()
    worst = numeric.idxmin() if higher_is_better else numeric.idxmax()
    return ['background-color: #c8e6c9' if i == best
            else 'background-color: #ffcdd2' if i == worst
            else '' for i in s.index]

styled = summary_df.style.apply(highlight_best).set_caption(
    'Table 1. 10-Fold Cross-Validation Performance Summary (green = best, red = worst)')
display(styled)
summary_df.to_csv('table01_cv_performance.csv')
print('Table 1 saved ✓')

## 7. Cross-Validation Score Distributions

In [ ]:
metrics_plot = ['R2', 'RMSE', 'MAE', 'MAPE']
metric_labels = ['R²', 'RMSE', 'MAE', 'MAPE (%)']
n_metrics = len(metrics_plot)

fig, axes = plt.subplots(1, n_metrics, figsize=(18, 5.5))

for ax, metric, mlabel in zip(axes, metrics_plot, metric_labels):
    data_box = [cv_results[n][metric] for n in MODEL_NAMES]
    bp = ax.boxplot(data_box, patch_artist=True, notch=False,
                    medianprops=dict(color='black', lw=2.5),
                    whiskerprops=dict(lw=1.5), capprops=dict(lw=1.5))
    colors_bp = list(PALETTE.values())
    for patch, col in zip(bp['boxes'], colors_bp):
        patch.set_facecolor(col)
        patch.set_alpha(0.65)
    # overlay strip
    for j, (nm, dat) in enumerate(zip(MODEL_NAMES, data_box), 1):
        jitter_pts = np.random.normal(j, 0.06, size=len(dat))
        ax.scatter(jitter_pts, dat, color=PALETTE[nm], s=22,
                   edgecolors='white', lw=0.5, zorder=3, alpha=0.8)
    ax.set_xticks(range(1, len(MODEL_NAMES)+1))
    ax.set_xticklabels([n.replace(' ', '\n') for n in MODEL_NAMES], fontsize=9)
    ax.set_title(mlabel, fontweight='bold')
    ax.set_xlabel('Model')

fig.suptitle('10-Fold Cross-Validation Score Distributions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig04_cv_score_distributions.png')
plt.show()
print('Figure 4 saved ✓')

## 8. Actual vs. Predicted Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

for ax, name in zip(axes, MODEL_NAMES):
    y_pred = oof_preds[name]
    r2     = r2_score(y, y_pred)
    rmse   = np.sqrt(mean_squared_error(y, y_pred))
    color  = PALETTE[name]

    ax.scatter(y, y_pred, alpha=0.45, s=18, c=color,
               edgecolors='white', lw=0.3, label='Samples')

    lo, hi = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=2, label='Perfect fit')

    # regression line
    m_fit, b_fit, r_val, _, _ = stats.linregress(y, y_pred)
    xr = np.linspace(lo, hi, 200)
    ax.plot(xr, m_fit*xr + b_fit, color='#E64B35', lw=2, ls='-', label='Reg. line')

    ax.set_xlabel('Actual Oil Recovery Factor (%)')
    ax.set_ylabel('Predicted Oil Recovery Factor (%)')
    ax.set_title(name, fontweight='bold')

    textstr = f'R²  = {r2:.4f}\nRMSE= {rmse:.4f}'
    ax.text(0.05, 0.93, textstr, transform=ax.transAxes, fontsize=10,
            va='top', bbox=dict(boxstyle='round', facecolor='white',
            alpha=0.85, edgecolor='grey'))
    ax.legend(fontsize=8)

fig.suptitle('Actual vs. Predicted — Out-of-Fold (10-Fold CV)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig05_actual_vs_predicted.png')
plt.show()
print('Figure 5 saved ✓')

## 9. Residual Analysis

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 18))

for row, name in enumerate(MODEL_NAMES):
    y_pred = oof_preds[name]
    resid  = y.values - y_pred
    color  = PALETTE[name]

    # --- Residuals vs Fitted
    ax = axes[row, 0]
    ax.scatter(y_pred, resid, alpha=0.4, s=15, c=color, edgecolors='white', lw=0.3)
    ax.axhline(0, color='black', lw=1.5, ls='--')
    lowess_x = pd.Series(y_pred).sort_values()
    ax.set_xlabel('Fitted Values')
    ax.set_ylabel('Residuals')
    ax.set_title(f'{name}\nResiduals vs Fitted', fontweight='bold', fontsize=10)

    # --- Residual distribution
    ax = axes[row, 1]
    sns.histplot(resid, kde=True, color=color, alpha=0.5, bins=35, ax=ax,
                 line_kws={'lw': 2})
    ax.axvline(0, color='black', lw=1.5, ls='--')
    stat_s, p_s = shapiro(resid[:5000] if len(resid) > 5000 else resid)
    ax.set_xlabel('Residuals')
    ax.set_ylabel('Count')
    ax.set_title(f'{name}\nResidual Dist. (Shapiro p={p_s:.3f})', fontweight='bold', fontsize=10)

    # --- Q-Q of residuals
    ax = axes[row, 2]
    (osm2, osr2), (sl2, int2, _) = probplot(resid, dist='norm')
    ax.scatter(osm2, osr2, color=color, alpha=0.5, s=15)
    lx = np.array([osm2.min(), osm2.max()])
    ax.plot(lx, sl2*lx + int2, color='black', lw=2)
    ax.set_xlabel('Theoretical Quantiles')
    ax.set_ylabel('Sample Quantiles')
    ax.set_title(f'{name}\nQ-Q Plot of Residuals', fontweight='bold', fontsize=10)

fig.suptitle('Residual Diagnostics — All Models',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig06_residual_analysis.png')
plt.show()
print('Figure 6 saved ✓')

## 10. Learning Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

train_sizes_pct = np.linspace(0.10, 1.0, 10)

for ax, name in zip(axes, MODEL_NAMES):
    model  = models[name]
    Xdata  = X_scaled.values if name == 'SVR' else X.values
    color  = PALETTE[name]

    train_sizes, train_scores, val_scores = learning_curve(
        model, Xdata, y.values,
        cv=5, n_jobs=-1, scoring='r2',
        train_sizes=train_sizes_pct
    )

    tr_mean  = train_scores.mean(axis=1)
    tr_std   = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std  = val_scores.std(axis=1)

    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.15, color=color)
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                    alpha=0.15, color='grey')
    ax.plot(train_sizes, tr_mean,  'o-', color=color,   lw=2, ms=5, label='Training R²')
    ax.plot(train_sizes, val_mean, 's--', color='grey', lw=2, ms=5, label='CV R²')

    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('R² Score')
    ax.set_title(name, fontweight='bold')
    ax.set_ylim([-0.05, 1.05])
    ax.legend()
    ax.axhline(1.0, color='green', ls=':', lw=1, alpha=0.4)

fig.suptitle('Learning Curves (5-Fold CV, Scoring: R²)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig07_learning_curves.png')
plt.show()
print('Figure 7 saved ✓')

## 11. AIC & BIC Comparison

In [ ]:
aic_vals, bic_vals = {}, {}
for name in MODEL_NAMES:
    resid   = y.values - oof_preds[name]
    n_obs   = len(y)
    k       = len(FEATURES) + 1
    sse     = np.sum(resid**2)
    sigma2  = sse / n_obs
    log_lik = -n_obs/2 * (1 + np.log(2*np.pi*sigma2))
    aic_vals[name] = 2*k - 2*log_lik
    bic_vals[name] = k*np.log(n_obs) - 2*log_lik

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

names  = MODEL_NAMES
colors = list(PALETTE.values())
x      = np.arange(len(names))

# --- AIC bar chart
ax = axes[0]
bars = ax.bar(x, [aic_vals[n] for n in names], color=colors, edgecolor='white',
               linewidth=0.7, width=0.55)
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
ax.set_ylabel('AIC')
ax.set_title('Akaike Information Criterion\n(lower = better)', fontweight='bold')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)

# --- BIC bar chart
ax = axes[1]
bars = ax.bar(x, [bic_vals[n] for n in names], color=colors, edgecolor='white',
               linewidth=0.7, width=0.55)
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
ax.set_ylabel('BIC')
ax.set_title('Bayesian Information Criterion\n(lower = better)', fontweight='bold')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)

# --- Joint AIC vs BIC scatter
ax = axes[2]
for nm, col in PALETTE.items():
    ax.scatter(aic_vals[nm], bic_vals[nm], s=200, color=col,
               edgecolors='black', lw=1.2, zorder=5, label=nm)
    ax.annotate(nm, (aic_vals[nm], bic_vals[nm]),
                textcoords='offset points', xytext=(8, 4), fontsize=8)
ax.set_xlabel('AIC')
ax.set_ylabel('BIC')
ax.set_title('AIC vs. BIC\n(bottom-left = better)', fontweight='bold')
ax.legend(fontsize=8)

fig.suptitle('Information Criteria — AIC & BIC',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig08_aic_bic.png')
plt.show()
print('Figure 8 saved ✓')

## 12. Feature Importance (Tree-based & Permutation)

In [ ]:
tree_models = ['Random Forest', 'XGBoost', 'Gradient Boosting']

fig, axes = plt.subplots(1, len(tree_models), figsize=(18, 6))

all_importance = {}
for ax, name in zip(axes, tree_models):
    model, Xdata = trained_models[name]
    importances  = model.feature_importances_
    feat_imp_df  = pd.DataFrame({'Feature': FEATURES, 'Importance': importances})
    feat_imp_df  = feat_imp_df.sort_values('Importance', ascending=True)
    all_importance[name] = feat_imp_df

    short_names = [f.split('(')[0].replace('_', ' ').strip() for f in feat_imp_df.Feature]
    cmap_imp = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_imp_df)))
    bars = ax.barh(range(len(feat_imp_df)), feat_imp_df.Importance,
                   color=cmap_imp, edgecolor='white', lw=0.4)
    ax.set_yticks(range(len(feat_imp_df)))
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Importance Score')
    ax.set_title(name, fontweight='bold')

fig.suptitle('Feature Importances — Tree-Based Models',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig09_feature_importance.png')
plt.show()
print('Figure 9 saved ✓')

# ── SVR permutation importance ─────────────────────────────────────────────
print('Computing SVR permutation importance ...')
svr_model, svr_Xdata = trained_models['SVR']
perm_imp = permutation_importance(svr_model, svr_Xdata, y.values,
                                   n_repeats=20, random_state=SEED, n_jobs=-1)

fig, ax = plt.subplots(figsize=(8, 6))
perm_df = pd.DataFrame({'Feature': FEATURES,
                         'Importance': perm_imp.importances_mean,
                         'Std': perm_imp.importances_std}).sort_values('Importance', ascending=True)
short_names = [f.split('(')[0].replace('_', ' ').strip() for f in perm_df.Feature]
colors_perm = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(perm_df)))
ax.barh(range(len(perm_df)), perm_df.Importance, xerr=perm_df.Std,
        color=colors_perm, edgecolor='white', lw=0.4, capsize=3)
ax.set_yticks(range(len(perm_df)))
ax.set_yticklabels(short_names, fontsize=9)
ax.set_xlabel('Mean Decrease in R²')
ax.set_title('SVR — Permutation Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig('fig10_svr_permutation_importance.png')
plt.show()
print('Figure 10 saved ✓')

## 13. SHAP Analysis

In [ ]:
# ── SHAP for each model ─────────────────────────────────────────────────────
shap_values_dict = {}

for name in MODEL_NAMES:
    model, Xdata = trained_models[name]
    Xdf = pd.DataFrame(Xdata, columns=FEATURES)

    if name == 'SVR':
        explainer = shap.KernelExplainer(
            model.predict,
            shap.kmeans(Xdf, 50)   # use k-means background for speed
        )
        # sample 300 rows for KernelExplainer
        idx_sample = np.random.choice(len(Xdf), min(300, len(Xdf)), replace=False)
        sv = explainer.shap_values(Xdf.iloc[idx_sample], silent=True)
        shap_values_dict[name] = (sv, Xdf.iloc[idx_sample])
    else:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(Xdf)
        shap_values_dict[name] = (sv, Xdf)

    print(f'  SHAP computed for {name} ✓')

print('All SHAP values computed ✓')

In [ ]:
# ── SHAP Summary Beeswarm ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes = axes.flatten()

for ax, name in zip(axes, MODEL_NAMES):
    sv, Xdf = shap_values_dict[name]
    plt.sca(ax)
    shap.summary_plot(sv, Xdf, plot_type='dot', show=False, max_display=14,
                      color_bar_label='Feature value',
                      plot_size=None)
    ax.set_title(f'{name} — SHAP Beeswarm', fontweight='bold', fontsize=11)

fig.suptitle('SHAP Summary Plots — All Models',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig11_shap_beeswarm.png')
plt.show()
print('Figure 11 saved ✓')

In [ ]:
# ── SHAP Mean Absolute Bar (all 4 models) ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

for ax, name in zip(axes, MODEL_NAMES):
    sv, Xdf = shap_values_dict[name]
    mean_abs = np.abs(sv).mean(axis=0)
    shap_df  = pd.DataFrame({'Feature': FEATURES, 'SHAP': mean_abs})
    shap_df  = shap_df.sort_values('SHAP', ascending=True)
    short    = [f.split('(')[0].replace('_', ' ').strip() for f in shap_df.Feature]
    cmap_s   = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(shap_df)))
    ax.barh(range(len(shap_df)), shap_df.SHAP, color=cmap_s,
            edgecolor='white', lw=0.4)
    ax.set_yticks(range(len(shap_df)))
    ax.set_yticklabels(short, fontsize=8)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f'{name}', fontweight='bold')

fig.suptitle('SHAP Mean Absolute Feature Importance',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig12_shap_bar.png')
plt.show()
print('Figure 12 saved ✓')

In [ ]:
# ── SHAP Dependence plots — top 3 features from best tree model ────────────
best_tree = 'XGBoost'
sv, Xdf   = shap_values_dict[best_tree]
mean_abs  = np.abs(sv).mean(axis=0)
top3_idx  = np.argsort(mean_abs)[::-1][:3]
top3_feat = [FEATURES[i] for i in top3_idx]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, feat in zip(axes, top3_feat):
    feat_idx = list(Xdf.columns).index(feat)
    shap.dependence_plot(
        feat_idx, sv, Xdf,
        interaction_index='auto',
        ax=ax, show=False
    )
    ax.set_title(feat.split('(')[0].replace('_', ' ').strip(), fontweight='bold', fontsize=10)

fig.suptitle(f'SHAP Dependence Plots ({best_tree} — Top 3 Features)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig13_shap_dependence.png')
plt.show()
print('Figure 13 saved ✓')

## 14. Radar / Spider Chart — Model Performance

In [ ]:
from matplotlib.patches import FancyArrowPatch

def radar_chart(ax, values, labels, title, color):
    N = len(labels)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    vals   = values + [values[0]]
    angs   = angles + [angles[0]]
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.plot(angs, vals, 'o-', lw=2, color=color)
    ax.fill(angs, vals, alpha=0.25, color=color)
    ax.set_thetagrids(np.degrees(angles), labels, fontsize=8)
    ax.set_title(title, size=10, fontweight='bold', pad=12)
    ax.set_ylim(0, 1)

# Normalise metrics to [0,1] (higher = better)
def normalise_metrics(cv_res):
    r2   = np.clip(cv_res['R2'].mean(), 0, 1)
    rmse = 1 - np.clip(cv_res['RMSE'].mean() / y.std(), 0, 1)
    mae  = 1 - np.clip(cv_res['MAE'].mean() / y.std(), 0, 1)
    mape = 1 - np.clip(cv_res['MAPE'].mean() / 100, 0, 1)
    stab = 1 - np.clip(cv_res['R2'].std(), 0, 1)
    return [r2, rmse, mae, mape, stab]

radar_labels = ['R²', 'RMSE\n(norm)', 'MAE\n(norm)', 'MAPE\n(norm)', 'Stability']

fig, axes = plt.subplots(1, 4, figsize=(18, 5),
                          subplot_kw=dict(polar=True))

for ax, name in zip(axes, MODEL_NAMES):
    vals = normalise_metrics(cv_results[name])
    radar_chart(ax, vals, radar_labels, name, PALETTE[name])

fig.suptitle('Radar Chart — Normalised Model Performance',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig14_radar_chart.png')
plt.show()
print('Figure 14 saved ✓')

## 15. Prediction Interval / Error Band Plot

In [ ]:
# Sort by actual value for a clean ribbon plot
sort_idx = np.argsort(y.values)
y_sorted = y.values[sort_idx]
x_axis   = np.arange(len(y_sorted))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, name in zip(axes, MODEL_NAMES):
    y_pred_sorted = oof_preds[name][sort_idx]
    resid_sorted  = y_sorted - y_pred_sorted
    mae_v = np.abs(resid_sorted).mean()

    ax.fill_between(x_axis,
                    y_pred_sorted - mae_v,
                    y_pred_sorted + mae_v,
                    alpha=0.25, color=PALETTE[name], label='±MAE band')
    ax.plot(x_axis, y_sorted,        'k-',  lw=1.5, alpha=0.7, label='Actual')
    ax.plot(x_axis, y_pred_sorted,   color=PALETTE[name],
            lw=1.5, ls='--', label='Predicted')
    ax.set_xlabel('Sample Index (sorted by actual)')
    ax.set_ylabel('Oil Recovery Factor (%)')
    ax.set_title(name, fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('Prediction Ribbons — Actual vs Predicted with ±MAE Band',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig15_prediction_ribbons.png')
plt.show()
print('Figure 15 saved ✓')

## 16. Fold-wise Performance Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric in zip(axes, ['R2', 'RMSE']):
    mat = pd.DataFrame(
        {n: cv_results[n][metric] for n in MODEL_NAMES},
        index=[f'Fold {i+1}' for i in range(10)]
    )
    cmap_h = 'RdYlGn' if metric == 'R2' else 'RdYlGn_r'
    sns.heatmap(mat, annot=True, fmt='.3f', cmap=cmap_h,
                linewidths=0.5, ax=ax, cbar_kws={'label': metric})
    ax.set_title(f'Per-Fold {metric} Heatmap', fontweight='bold')
    ax.set_xlabel('Model')
    ax.set_ylabel('Fold')

fig.suptitle('Fold-wise Performance Heatmap (10-Fold CV)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig16_fold_heatmap.png')
plt.show()
print('Figure 16 saved ✓')

## 17. KDE of Predictions vs Actual

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, name in zip(axes, MODEL_NAMES):
    y_pred = oof_preds[name]
    sns.kdeplot(y,      ax=ax, fill=True,  alpha=0.35, color='#3C5488',
                label='Actual', linewidth=2.5)
    sns.kdeplot(y_pred, ax=ax, fill=True,  alpha=0.35, color=PALETTE[name],
                label='Predicted', linewidth=2.5)
    ax.set_xlabel('Oil Recovery Factor (%)')
    ax.set_ylabel('Density')
    ax.set_title(name, fontweight='bold')
    ax.legend()
    # KS test
    ks_stat, ks_p = stats.ks_2samp(y.values, y_pred)
    ax.text(0.98, 0.95, f'KS stat={ks_stat:.3f}\np={ks_p:.3f}',
            transform=ax.transAxes, fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.suptitle('KDE: Actual vs. Predicted Distribution per Model',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig17_kde_actual_vs_predicted.png')
plt.show()
print('Figure 17 saved ✓')

## 18. Cumulative Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- (a) ECDF of absolute error
ax = axes[0]
for name in MODEL_NAMES:
    abs_err = np.abs(y.values - oof_preds[name])
    sorted_err = np.sort(abs_err)
    ecdf = np.arange(1, len(sorted_err)+1) / len(sorted_err)
    ax.plot(sorted_err, ecdf, lw=2, color=PALETTE[name], label=name)
ax.set_xlabel('Absolute Error')
ax.set_ylabel('Cumulative Probability')
ax.set_title('(a) ECDF of Absolute Error', fontweight='bold')
ax.legend()
ax.axvline(0.5, color='grey', ls=':', lw=1)

# --- (b) Percentage within threshold
ax = axes[1]
thresholds = np.linspace(0, y.std()*2, 100)
for name in MODEL_NAMES:
    abs_err = np.abs(y.values - oof_preds[name])
    pct_within = [(abs_err <= t).mean()*100 for t in thresholds]
    ax.plot(thresholds, pct_within, lw=2, color=PALETTE[name], label=name)
ax.set_xlabel('Error Threshold')
ax.set_ylabel('% Predictions Within Threshold')
ax.set_title('(b) % Predictions Within Error Threshold', fontweight='bold')
ax.axhline(90, color='grey', ls=':', lw=1, label='90%')
ax.legend(fontsize=8)

fig.suptitle('Cumulative Error Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig18_cumulative_error.png')
plt.show()
print('Figure 18 saved ✓')

## 19. Feature Correlation with Target

In [ ]:
pearson_target = {}
spearman_target = {}
pvals = {}
for feat in FEATURES:
    r_p, p_p = pearsonr(df[feat], df[TARGET])
    r_s, p_s = stats.spearmanr(df[feat], df[TARGET])
    pearson_target[feat]  = r_p
    spearman_target[feat] = r_s
    pvals[feat] = p_p

corr_df = pd.DataFrame({
    'Pearson r' : pearson_target,
    'Spearman r': spearman_target,
    'p-value'   : pvals
}).sort_values('Pearson r', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, metric in zip(axes, ['Pearson r', 'Spearman r']):
    vals   = corr_df[metric]
    cols   = ['#E64B35' if v < 0 else '#4DBBD5' for v in vals]
    short  = [f.split('(')[0].replace('_', ' ').strip() for f in corr_df.index]
    bars   = ax.barh(range(len(vals)), vals, color=cols, edgecolor='white', lw=0.4)
    # significance stars
    for i, (feat, bar) in enumerate(zip(corr_df.index, bars)):
        pv = pvals[feat]
        star = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
        xpos = bar.get_width() + (0.01 if bar.get_width() >= 0 else -0.01)
        ax.text(xpos, i, star, va='center', fontsize=8, color='black')
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels(short, fontsize=9)
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel(f'{metric} with Oil Recovery Factor')
    ax.set_title(metric, fontweight='bold')
    ax.set_xlim(-1, 1.1)

neg_patch = mpatches.Patch(color='#E64B35', label='Negative correlation')
pos_patch = mpatches.Patch(color='#4DBBD5', label='Positive correlation')
fig.legend(handles=[pos_patch, neg_patch], loc='upper right', fontsize=9)
fig.suptitle('Feature Correlation with Oil Recovery Factor\n(* p<0.05, ** p<0.01, *** p<0.001)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig19_feature_target_correlation.png')
plt.show()
print('Figure 19 saved ✓')

## 20. Scatter Matrix — Top Features

In [ ]:
# Top 5 features by absolute Pearson with target
top5 = corr_df['Pearson r'].abs().nlargest(5).index.tolist()
pair_df = df[top5 + [TARGET]].copy()
short_cols = [c.split('(')[0].replace('_', ' ').strip() for c in pair_df.columns]
pair_df.columns = short_cols

g = sns.pairplot(pair_df, diag_kind='kde', plot_kws={'alpha': 0.35, 's': 15,
                 'color': '#4DBBD5', 'edgecolors': 'white'},
                 diag_kws={'fill': True, 'color': '#3C5488', 'linewidth': 1.5})
g.fig.suptitle('Pair Plot — Top 5 Features + Oil Recovery Factor',
               fontsize=13, fontweight='bold', y=1.01)
plt.savefig('fig20_scatter_matrix.png')
plt.show()
print('Figure 20 saved ✓')

## 21. Taylor Diagram

In [ ]:
def taylor_diagram(ax, y_obs, y_preds_dict, palette):
    std_obs = y_obs.std()
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_thetamin(0)
    ax.set_thetamax(90)

    # Correlation arcs
    corr_ticks = [0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
    for r in corr_ticks:
        theta = np.arccos(r)
        ax.axvline(theta, color='grey', lw=0.5, alpha=0.5)
        ax.text(theta, ax.get_rmax()*1.05, f'{r:.2f}',
                ha='center', va='bottom', fontsize=7, color='grey')

    # Standard deviation arcs
    max_std = max([np.std(v) for v in y_preds_dict.values()] + [std_obs]) * 1.2
    ax.set_rmax(max_std)
    r_ticks = np.linspace(0, max_std, 5)
    ax.set_rticks(r_ticks)

    # RMSE contours
    theta_vals = np.linspace(0, np.pi/2, 200)
    for rmse_c in np.linspace(std_obs*0.25, std_obs*1.5, 5):
        r_contour = np.sqrt(std_obs**2 + rmse_c**2 - 2*std_obs*rmse_c*np.cos(theta_vals))
        ax.plot(theta_vals, r_contour, 'k:', lw=0.6, alpha=0.3)

    # Reference point
    ax.plot(0, std_obs, 'k*', ms=12, label='Observation', zorder=10)

    for name, y_pred in y_preds_dict.items():
        r_val = np.corrcoef(y_obs, y_pred)[0, 1]
        std_p = np.std(y_pred)
        theta = np.arccos(r_val)
        ax.scatter(theta, std_p, s=120, color=palette[name],
                   edgecolors='black', lw=1, zorder=8, label=name)
        ax.text(theta+0.02, std_p+max_std*0.02, name.replace(' ', '\n'),
                fontsize=7)

    ax.set_xlabel('Correlation', labelpad=20)

fig = plt.figure(figsize=(8, 8))
ax  = fig.add_subplot(111, polar=True)
taylor_diagram(ax, y.values,
               {n: oof_preds[n] for n in MODEL_NAMES},
               PALETTE)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax.set_title('Taylor Diagram — Model Performance Summary',
             fontweight='bold', pad=20)
plt.savefig('fig21_taylor_diagram.png', bbox_inches='tight')
plt.show()
print('Figure 21 saved ✓')

## 22. Combined Metric Violin Plot

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5.5))

for ax, (metric, mlabel) in zip(axes, zip(['R2', 'RMSE', 'MAE', 'MAPE'],
                                            ['R²', 'RMSE', 'MAE', 'MAPE (%)'])):
    violin_data = [cv_results[n][metric] for n in MODEL_NAMES]
    parts = ax.violinplot(violin_data, positions=range(len(MODEL_NAMES)),
                          showmeans=True, showmedians=True, showextrema=True)
    for i, (pc, nm) in enumerate(zip(parts['bodies'], MODEL_NAMES)):
        pc.set_facecolor(PALETTE[nm])
        pc.set_alpha(0.6)
    parts['cmeans'].set_color('black')
    parts['cmedians'].set_color('#E64B35')

    for j, (nm, dat) in enumerate(zip(MODEL_NAMES, violin_data)):
        jit = np.random.normal(j, 0.04, len(dat))
        ax.scatter(jit, dat, color=PALETTE[nm], s=25,
                   edgecolors='white', lw=0.4, zorder=3, alpha=0.8)

    ax.set_xticks(range(len(MODEL_NAMES)))
    ax.set_xticklabels([n.replace(' ', '\n') for n in MODEL_NAMES], fontsize=8)
    ax.set_title(mlabel, fontweight='bold')
    ax.set_xlabel('Model')

fig.suptitle('Violin Plots — 10-Fold CV Metric Distributions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig22_violin_metrics.png')
plt.show()
print('Figure 22 saved ✓')

## 23. Statistical Significance — Friedman + Wilcoxon Tests

In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

r2_arrays = [cv_results[n]['R2'] for n in MODEL_NAMES]

# Friedman test
friedman_stat, friedman_p = friedmanchisquare(*r2_arrays)
print(f'Friedman test: stat={friedman_stat:.4f}, p={friedman_p:.4f}')
if friedman_p < 0.05:
    print('  → Significant differences among models (p < 0.05)')
else:
    print('  → No significant differences among models')

# Pairwise Wilcoxon signed-rank tests
pairs = list(combinations(MODEL_NAMES, 2))
wilcoxon_results = []
for m1, m2 in pairs:
    try:
        stat, p = wilcoxon(cv_results[m1]['R2'], cv_results[m2]['R2'])
    except ValueError:
        stat, p = np.nan, np.nan
    wilcoxon_results.append({'Model 1': m1, 'Model 2': m2,
                              'Statistic': f'{stat:.3f}', 'p-value': f'{p:.4f}',
                              'Significant': 'Yes' if p < 0.05 else 'No'})

wilcoxon_df = pd.DataFrame(wilcoxon_results)
wilcoxon_df.to_csv('table02_wilcoxon_tests.csv', index=False)

# Build p-value heatmap
pmat = pd.DataFrame(np.ones((4, 4)), index=MODEL_NAMES, columns=MODEL_NAMES)
for row in wilcoxon_results:
    pmat.loc[row['Model 1'], row['Model 2']] = float(row['p-value'])
    pmat.loc[row['Model 2'], row['Model 1']] = float(row['p-value'])

fig, ax = plt.subplots(figsize=(8, 6))
mask_diag = np.eye(4, dtype=bool)
sns.heatmap(pmat, annot=True, fmt='.3f', cmap='RdYlGn_r', vmin=0, vmax=0.1,
            mask=mask_diag, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'p-value'})
xnames = [n.replace(' ', '\n') for n in MODEL_NAMES]
ax.set_xticklabels(xnames)
ax.set_yticklabels(MODEL_NAMES, rotation=0)
ax.set_title('Pairwise Wilcoxon Signed-Rank Test\n(p-values, R² across folds)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('fig23_statistical_significance.png')
plt.show()
print('Figure 23 saved ✓')
display(wilcoxon_df)

## 24. Final Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── R² comparison ──────────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, :2])
means = [cv_results[n]['R2'].mean() for n in MODEL_NAMES]
stds  = [cv_results[n]['R2'].std()  for n in MODEL_NAMES]
bars  = ax0.bar(MODEL_NAMES, means, yerr=stds, capsize=6,
                color=list(PALETTE.values()), edgecolor='white',
                error_kw=dict(ecolor='black', lw=1.5, capthick=1.5))
ax0.set_ylim([0, 1.05])
ax0.set_ylabel('Mean R²')
ax0.set_title('Mean R² ± Std (10-Fold CV)', fontweight='bold')
ax0.set_xticklabels([n.replace(' ', '\n') for n in MODEL_NAMES], fontsize=9)
for bar, m, s in zip(bars, means, stds):
    ax0.text(bar.get_x()+bar.get_width()/2, bar.get_height()+s+0.01,
             f'{m:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── RMSE comparison ────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 2:])
rmse_means = [cv_results[n]['RMSE'].mean() for n in MODEL_NAMES]
rmse_stds  = [cv_results[n]['RMSE'].std()  for n in MODEL_NAMES]
ax1.bar(MODEL_NAMES, rmse_means, yerr=rmse_stds, capsize=6,
        color=list(PALETTE.values()), edgecolor='white',
        error_kw=dict(ecolor='black', lw=1.5, capthick=1.5))
ax1.set_ylabel('Mean RMSE')
ax1.set_title('Mean RMSE ± Std (10-Fold CV)', fontweight='bold')
ax1.set_xticklabels([n.replace(' ', '\n') for n in MODEL_NAMES], fontsize=9)

# ── Best model actual vs predicted ────────────────────────────────────────
best_model = MODEL_NAMES[np.argmax(means)]
ax2 = fig.add_subplot(gs[1, :2])
y_pred_best = oof_preds[best_model]
ax2.scatter(y, y_pred_best, alpha=0.4, s=15,
            c=PALETTE[best_model], edgecolors='white', lw=0.3)
lo2, hi2 = y.min(), y.max()
ax2.plot([lo2, hi2], [lo2, hi2], 'k--', lw=2)
ax2.set_xlabel('Actual')
ax2.set_ylabel('Predicted')
ax2.set_title(f'Best Model: {best_model}\nActual vs Predicted', fontweight='bold')
ax2.text(0.05, 0.93, f'R²={r2_score(y, y_pred_best):.4f}',
         transform=ax2.transAxes, fontsize=10, va='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

# ── SHAP bar best model ─────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 2:])
sv_b, Xdf_b = shap_values_dict[best_model]
mean_abs_b = np.abs(sv_b).mean(axis=0)
shap_df_b  = pd.DataFrame({'Feature': FEATURES, 'SHAP': mean_abs_b})
shap_df_b  = shap_df_b.sort_values('SHAP', ascending=True).tail(10)
short_b    = [f.split('(')[0].replace('_', ' ').strip() for f in shap_df_b.Feature]
cmap_sb    = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(shap_df_b)))
ax3.barh(range(len(shap_df_b)), shap_df_b.SHAP, color=cmap_sb)
ax3.set_yticks(range(len(shap_df_b)))
ax3.set_yticklabels(short_b, fontsize=8)
ax3.set_xlabel('Mean |SHAP value|')
ax3.set_title(f'{best_model} — Top 10 SHAP Features', fontweight='bold')

# ── Metric comparison table ────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, :])
ax4.axis('off')
table_data = []
col_labels = ['Model', 'R² (mean±std)', 'RMSE (mean±std)', 'MAE (mean±std)', 'MAPE % (mean)', 'AIC', 'BIC']
for name in MODEL_NAMES:
    res = cv_results[name]
    resid   = y.values - oof_preds[name]
    n_obs   = len(y); k = len(FEATURES)+1
    sse     = np.sum(resid**2)
    sigma2  = sse/n_obs
    log_lik = -n_obs/2*(1+np.log(2*np.pi*sigma2))
    aic_v   = 2*k - 2*log_lik
    bic_v   = k*np.log(n_obs) - 2*log_lik
    table_data.append([
        name,
        f"{res['R2'].mean():.4f}±{res['R2'].std():.4f}",
        f"{res['RMSE'].mean():.4f}±{res['RMSE'].std():.4f}",
        f"{res['MAE'].mean():.4f}±{res['MAE'].std():.4f}",
        f"{res['MAPE'].mean():.2f}",
        f"{aic_v:.1f}",
        f"{bic_v:.1f}",
    ])

tbl = ax4.table(cellText=table_data, colLabels=col_labels,
                cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#F2F2F2')
    cell.set_edgecolor('white')

fig.suptitle('Summary Dashboard — Comparative ML Analysis for Oil Recovery Prediction',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('fig24_summary_dashboard.png', bbox_inches='tight')
plt.show()
print('Figure 24 saved ✓')
print(f'\n★ Best model: {best_model} with R²={max(means):.4f}')

## 25. Export All Figures Summary

In [ ]:
import os

figs = sorted([f for f in os.listdir('.') if f.startswith('fig') and f.endswith('.png')])
tabs = sorted([f for f in os.listdir('.') if f.startswith('table') and f.endswith('.csv')])

print('=' * 60)
print('  FIGURES SAVED')
print('=' * 60)
for f in figs:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f:<45} {size_kb:>6.1f} KB')

print()
print('=' * 60)
print('  TABLES SAVED')
print('=' * 60)
for t in tabs:
    print(f'  {t}')

print()
print('═' * 60)
print('  FINAL PERFORMANCE RANKING (by mean R²)')
print('═' * 60)
ranking = sorted(MODEL_NAMES, key=lambda n: cv_results[n]['R2'].mean(), reverse=True)
for rank, name in enumerate(ranking, 1):
    r2m  = cv_results[name]['R2'].mean()
    rmse = cv_results[name]['RMSE'].mean()
    print(f'  #{rank}  {name:<25}  R²={r2m:.4f}   RMSE={rmse:.4f}')
print('═' * 60)